# B3：文件智能處理 Pipeline — 合約 / 發票 / 報告

> **場景：** 台灣大型金融機構每月處理 5 萬份合約文件，需要自動抽取條款、偵測風險條款、支援自然語言查詢。  
> **核心挑戰：** 不同類型的文件有不同的結構，直接 chunk 會破壞合約條款的語意。

## 核心架構決策

| 決策 | 選擇 | 拒絕的方案 |
|------|------|------------|
| Chunking | Parent-Child（小塊檢索，大塊合成） | Fixed-Size（破壞條款語意） |
| Metadata | 豐富 metadata + 過濾搜尋 | 只存內容向量 |
| 版本管理 | 版本號 + 時間戳 metadata | 覆蓋舊版本 |
| 多文件推理 | Map-Reduce | 把所有文件塞進一次 Context |
| 結構提取 | 結構化 Extraction + 驗證 | 讓 LLM 自由格式輸出 |

In [ ]:
import re
import json
import time
import asyncio
import uuid
from typing import Optional
from dataclasses import dataclass, field

try:
    from dotenv import load_dotenv; load_dotenv()
    from langchain_openai import ChatOpenAI
    from langchain_core.messages import HumanMessage, SystemMessage
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    LLM_AVAILABLE = True
    print("✅ LLM 可用")
except:
    LLM_AVAILABLE = False
    print("⚠️  LLM 未設定")

# 模擬合約文件
SAMPLE_CONTRACT = """
委託服務合約書

立約人：
  委託方（甲方）：台灣銀行股份有限公司
  受託方（乙方）：科技顧問有限公司

第一條 服務內容
乙方受甲方委託，提供企業數位轉型顧問服務，包含系統分析、架構設計及導入輔導。
服務期間自民國115年1月1日起至民國115年12月31日止，共計12個月。

第二條 服務費用
本合約服務費用總計新台幣壹千伍百萬元整（NTD 15,000,000），
分三期付款：
  第一期：簽約後7日內，金額新台幣伍佰萬元整（NTD 5,000,000）
  第二期：服務進行至第六個月，金額新台幣伍佰萬元整（NTD 5,000,000）
  第三期：服務完成驗收後，金額新台幣伍佰萬元整（NTD 5,000,000）

第三條 保密義務
乙方對於執行本合約所獲知之甲方營業秘密及客戶資料，應負保密義務。
保密期間為合約期間及合約終止後五年。違反保密義務者，應賠償損害。

第四條 違約責任
任一方違反本合約時，應賠償他方所受損害。
甲方若未按期付款，應加計年息百分之十（10%）之遲延利息。
乙方若未按期交付成果，每日按合約金額千分之一（0.1%）計算違約金，
最高以合約總額百分之十（10%）為限，即新台幣壹佰伍拾萬元整（NTD 1,500,000）。

第五條 準據法及管轄
本合約依中華民國法律為準據法。如有爭議，雙方同意以台北地方法院為第一審管轄法院。
"""

print(f"合約字數：{len(SAMPLE_CONTRACT)} 字")

## 決策 1：Parent-Child Chunking — 為什麼不用 Fixed-Size？

**❌ 被拒絕：Fixed-Size Chunking（512 tokens）**
```
Chunk 1: ...第四條 違約責任\n任一方違反本合約時...（被切斷）
Chunk 2: ...甲方若未按期付款，應加計年息百分之十（10%）...
Chunk 3: ...乙方若未按期交付成果，每日按合約金額千分...
```
問題：「第四條的違約金上限是多少？」→ 需要拼接 Chunk 2+3，但它們可能沒有一起被檢索到。

**✅ 選擇：Parent-Child Chunking — 小塊索引，大塊上下文**

In [ ]:
@dataclass
class DocumentChunk:
    chunk_id: str
    parent_id: Optional[str]    # None = Parent chunk
    doc_id: str
    content: str
    chunk_type: str             # parent | child | sentence
    metadata: dict = field(default_factory=dict)


class ParentChildChunker:
    """
    Parent-Child Chunking：
    - Child chunks（小）：用於 Embedding + 向量搜尋（精確定位）
    - Parent chunks（大）：用於 LLM Context（完整語意）
    
    為什麼這樣設計？
    - Child 小：Embedding 更精確（不稀釋關鍵詞）
    - Parent 大：LLM 看到的是完整條款（不缺上下文）
    
    具體做法：
    1. 按「條」切割成 Parent chunks（每條一個）
    2. 每個 Parent 再切成 2-3 個 Child chunks
    3. 搜尋時用 Child，回傳給 LLM 時換成 Parent
    """
    
    def __init__(self, parent_size: int = 800, child_size: int = 200):
        self.parent_size = parent_size
        self.child_size = child_size
    
    def chunk_contract(self, doc_id: str, text: str, metadata: dict = None) -> list[DocumentChunk]:
        """按合約條款結構切割"""
        chunks = []
        base_meta = metadata or {}
        
        # 按「第X條」切割成 Parent chunks
        article_pattern = re.compile(r'(第[一二三四五六七八九十\d]+條.*?)(?=第[一二三四五六七八九十\d]+條|$)',
                                      re.DOTALL)
        articles = article_pattern.findall(text)
        
        if not articles:
            # 如果沒有條款結構，退回到段落切割
            articles = [p for p in text.split('\n\n') if p.strip()]
        
        for i, article_text in enumerate(articles):
            article_text = article_text.strip()
            if not article_text:
                continue
            
            # 提取條款標題
            title_match = re.match(r'(第[一二三四五六七八九十\d]+條[^\n]*)', article_text)
            article_title = title_match.group(1) if title_match else f"段落 {i+1}"
            
            parent_id = f"{doc_id}_art_{i+1}"
            
            # 建立 Parent chunk
            parent = DocumentChunk(
                chunk_id=parent_id,
                parent_id=None,
                doc_id=doc_id,
                content=article_text,
                chunk_type="parent",
                metadata={**base_meta, "article": article_title, "article_num": i+1}
            )
            chunks.append(parent)
            
            # 建立 Child chunks（句子級別）
            sentences = re.split(r'[。；]', article_text)
            for j, sentence in enumerate(sentences):
                sentence = sentence.strip()
                if len(sentence) < 10:  # 太短的跳過
                    continue
                child = DocumentChunk(
                    chunk_id=f"{parent_id}_s{j+1}",
                    parent_id=parent_id,
                    doc_id=doc_id,
                    content=sentence,
                    chunk_type="child",
                    metadata={**base_meta, "article": article_title, "sentence_num": j+1}
                )
                chunks.append(child)
        
        return chunks


chunker = ParentChildChunker()
doc_id = "contract_001"
metadata = {
    "doc_type": "service_contract",
    "party_a": "台灣銀行",
    "party_b": "科技顧問有限公司",
    "contract_value": 15_000_000,
    "version": "1.0",
    "created_at": time.strftime("%Y-%m-%d")
}

chunks = chunker.chunk_contract(doc_id, SAMPLE_CONTRACT, metadata)

parents = [c for c in chunks if c.chunk_type == "parent"]
children = [c for c in chunks if c.chunk_type == "child"]

print("Parent-Child Chunking 結果：")
print("=" * 55)
print(f"Parent chunks（用於 LLM Context）: {len(parents)} 個")
print(f"Child chunks（用於向量搜尋）: {len(children)} 個")
print()
for parent in parents:
    children_of_parent = [c for c in children if c.parent_id == parent.chunk_id]
    print(f"  Parent: {parent.metadata['article']}")
    print(f"    內容：{parent.content[:80]}...")
    print(f"    Child chunks: {len(children_of_parent)} 個")
    for child in children_of_parent[:2]:
        print(f"      → {child.content[:50]}")
    print()

print("\n⭐ 搜尋流程：")
print("  1. 用戶問：『違約金最高是多少？』")
print("  2. 向量搜尋 Child chunks → 找到 child: 違約金相關句子")
print("  3. 換成 Parent（第四條 完整文本）給 LLM")
print("  4. LLM 看到完整條款，答案準確")

## 決策 2：豐富 Metadata — 為什麼 Metadata 比向量更重要？

In [ ]:
# Metadata 過濾 vs 純向量搜尋

print("""
為什麼 Metadata 設計比 Embedding 模型更重要？
═══════════════════════════════════════════════════

情境：法務問「2026 年服務合約中違約金超過 1000 萬的有哪些？」

❌ 純向量搜尋：
  向量搜尋「違約金超過 1000 萬」→
  可能找到所有年份、所有類型合約的違約條款
  → 需要人工再過濾

✅ Metadata 過濾 + 向量搜尋：
  先 filter：doc_type='service_contract' AND year=2026
             AND contract_value > 10_000_000
  再向量搜尋：在過濾後的子集中找「違約金」
  → 精確、快速、不會有跨年度的干擾

必要的 Metadata 欄位設計：
""")

METADATA_SCHEMA = {
    # 文件識別
    "doc_id": "唯一識別碼",
    "doc_type": "合約類型：service/purchase/nda/lease",
    "version": "版本號，用於版本管理",
    "is_latest": "是否為最新版本（布林值）",
    
    # 時間
    "created_at": "建立日期 YYYY-MM-DD",
    "effective_date": "合約生效日",
    "expiry_date": "合約到期日",
    "year": "年份（用於過濾）",
    
    # 當事方
    "party_a": "甲方名稱",
    "party_b": "乙方名稱",
    
    # 財務
    "contract_value": "合約總金額（數值，用於範圍過濾）",
    "currency": "幣別",
    
    # 風險
    "risk_flags": "list of 風險標記：[no_penalty_cap, missing_termination_clause]",
    "reviewed_by": "法務審閱人",
    "review_status": "approved/pending/rejected",
    
    # 結構
    "article": "所屬條款（第X條）",
    "article_num": "條款號碼（用於排序）",
}

for field_name, description in METADATA_SCHEMA.items():
    print(f"  {field_name:<20}: {description}")

print("\n\n常見查詢 → Metadata 過濾策略：")
filter_examples = [
    ("今年到期的合約",           {"year": 2026, "is_latest": True}),
    ("違約金超過 500 萬的條款",   {"doc_type": "service", "contract_value": ">5000000"}),
    ("待審閱的保密協議",          {"doc_type": "nda", "review_status": "pending"}),
    ("第四條的所有版本",          {"article_num": 4, "doc_id": "contract_001"}),
]
for query, filters in filter_examples:
    print(f"  '{query}' → {filters}")

## 決策 3：結構化資訊提取 + 驗證

In [ ]:
from dataclasses import dataclass
from typing import List, Optional

@dataclass
class ContractExtraction:
    """結構化提取的合約資訊"""
    parties: dict
    service_period: dict
    payment_terms: list
    penalty_clauses: list
    risk_flags: list
    total_value: Optional[float] = None

EXTRACTION_SCHEMA = """
從合約中提取以下資訊，以 JSON 格式輸出：
{
  "parties": {
    "party_a": "甲方名稱",
    "party_b": "乙方名稱"
  },
  "service_period": {
    "start_date": "YYYY-MM-DD",
    "end_date": "YYYY-MM-DD",
    "duration_months": 整數
  },
  "total_value": 數字（台幣），
  "payment_terms": [
    {"period": "第一期", "amount": 數字, "trigger": "觸發條件"}
  ],
  "penalty_clauses": [
    {"party": "甲方/乙方", "condition": "觸發條件", "rate": "利率/比例", "cap": "上限金額"}
  ],
  "risk_flags": ["潛在風險列表"]
}
"""

async def extract_contract_info(contract_text: str) -> dict:
    """提取合約結構化資訊"""
    if LLM_AVAILABLE:
        prompt = f"{EXTRACTION_SCHEMA}\n\n合約內容：\n{contract_text}"
        response = await llm.ainvoke([
            SystemMessage(content="你是一個專業的合約分析 AI。只輸出 JSON，不要其他文字。"),
            HumanMessage(content=prompt)
        ])
        raw = response.content.strip()
        if "```" in raw:
            raw = raw.split("```")[1]
            if raw.startswith("json"): raw = raw[4:]
        try:
            return json.loads(raw.strip())
        except:
            return {"error": "JSON 解析失敗", "raw": raw[:200]}
    else:
        # 模擬提取結果
        return {
            "parties": {"party_a": "台灣銀行", "party_b": "科技顧問有限公司"},
            "service_period": {"start_date": "2026-01-01", "end_date": "2026-12-31", "duration_months": 12},
            "total_value": 15000000,
            "payment_terms": [
                {"period": "第一期", "amount": 5000000, "trigger": "簽約後7日內"},
                {"period": "第二期", "amount": 5000000, "trigger": "服務進行至第六個月"},
                {"period": "第三期", "amount": 5000000, "trigger": "服務完成驗收後"}
            ],
            "penalty_clauses": [
                {"party": "甲方", "condition": "未按期付款", "rate": "10%年息", "cap": None},
                {"party": "乙方", "condition": "未按期交付", "rate": "每日0.1%", "cap": 1500000}
            ],
            "risk_flags": ["甲方違約無上限（只有乙方有上限）", "保密義務合約後5年"]
        }


def validate_extraction(data: dict) -> dict:
    """
    驗證提取結果的完整性和合理性
    為什麼要驗證？
    → LLM 有時候會漏掉欄位或輸出格式不一致
    → 不驗證就存入 DB，後續查詢會 NullPointerError
    """
    issues = []
    
    # 必填欄位
    required = ["parties", "total_value", "payment_terms", "service_period"]
    for field in required:
        if field not in data or data[field] is None:
            issues.append(f"缺少必填欄位：{field}")
    
    # 金額驗證
    if "payment_terms" in data and "total_value" in data:
        if data["total_value"] and data["payment_terms"]:
            payment_sum = sum(t.get("amount", 0) for t in data["payment_terms"])
            if abs(payment_sum - data["total_value"]) > 1000:  # 允許小誤差
                issues.append(f"付款總額 ({payment_sum:,}) 與合約金額 ({data['total_value']:,}) 不符")
    
    return {"valid": len(issues) == 0, "issues": issues}


print("合約資訊結構化提取：")
extracted = await extract_contract_info(SAMPLE_CONTRACT)

if "error" not in extracted:
    print(json.dumps(extracted, ensure_ascii=False, indent=2))
    
    validation = validate_extraction(extracted)
    print(f"\n驗證結果：{'✅ 通過' if validation['valid'] else '❌ 有問題'}")
    if validation["issues"]:
        for issue in validation["issues"]:
            print(f"  ⚠️  {issue}")
    
    # 自動偵測風險標記
    print("\n⚠️  自動偵測到的風險：")
    for flag in extracted.get("risk_flags", []):
        print(f"  🔴 {flag}")
else:
    print(f"提取錯誤：{extracted}")

## 決策 4：Multi-Document 推理 — Map-Reduce vs 全塞 Context

In [ ]:
# 模擬多份合約的分析（Map-Reduce 模式）

CONTRACTS = [
    {"id": "C001", "content": "第四條：違約金上限為合約總額 10%，即 150 萬元。"},
    {"id": "C002", "content": "第五條：違約罰款無上限，按實際損失賠償。"},
    {"id": "C003", "content": "第三條：違約金為每日合約金額 0.5%，不設上限。"},
]

async def map_reduce_analysis(contracts: list[dict], query: str) -> str:
    """
    Map-Reduce 模式：
    Map：對每份合約單獨分析
    Reduce：彙整所有分析結果
    
    為什麼不把所有合約塞進一次 Context？
    → 50 份合約 × 5,000 tokens = 250K tokens
    → 超過大部分模型的 context window
    → 即使有 1M context 的 Gemini，Lost-in-the-Middle 效應會影響品質
    → 成本：每次查詢都付 250K tokens 的費用
    
    Map-Reduce 的成本：
    → Map：N 次 × 5,000 tokens（個別分析）
    → Reduce：1 次 × N × 摘要 tokens
    → 總成本通常低很多，且可以並行（降低延遲）
    """
    
    print(f"\nMap-Reduce 分析：'{query}'")
    print(f"合約數量：{len(contracts)}")
    
    # === MAP 階段：並行分析每份合約 ===
    async def map_single(contract: dict) -> dict:
        if LLM_AVAILABLE:
            resp = await llm.ainvoke([HumanMessage(
                content=f"針對以下問題分析這份合約，用1-2句話：\n問題：{query}\n合約（{contract['id']}）：{contract['content']}"
            )])
            analysis = resp.content
        else:
            analysis = f"[{contract['id']}] 違約條款分析：{contract['content'][:50]}"
        return {"id": contract["id"], "analysis": analysis}
    
    print("\n[Map] 並行分析各合約...")
    map_results = await asyncio.gather(*[map_single(c) for c in contracts])
    
    for r in map_results:
        print(f"  {r['id']}: {r['analysis'][:60]}")
    
    # === REDUCE 階段：彙整所有分析 ===
    combined = "\n".join(f"- {r['id']}: {r['analysis']}" for r in map_results)
    
    if LLM_AVAILABLE:
        reduce_resp = await llm.ainvoke([HumanMessage(
            content=f"以下是各合約的分析，請彙整回答問題：\n問題：{query}\n\n{combined}"
        )])
        final_answer = reduce_resp.content
    else:
        final_answer = f"綜合分析：C002 和 C003 存在高風險（無上限違約條款），C001 有合理上限。"
    
    print(f"\n[Reduce] 彙整結論：")
    print(f"  {final_answer}")
    
    return final_answer


answer = await map_reduce_analysis(
    CONTRACTS,
    "哪些合約的違約條款對我們（甲方）風險最高？請比較並排序。"
)

print("""

Map-Reduce vs 全部塞 Context：
  50 份合約全塞：250K tokens × $0.00015 = $0.038 / 次查詢
  Map-Reduce：  50 次 × 5K tokens + 1 次 × 2K = 252K tokens
               但可以並行！延遲從 50s 降到 max(每份時間)
  
關鍵：Map 階段可以完全並行，Reduce 只做一次彙整
""")

In [ ]:
print("""
FDE 面試：文件智能處理架構決策
═══════════════════════════════════════════════════════

Q: 為什麼用 Parent-Child Chunking，不用 Fixed-Size？
A: 合約有固有的語意單位——「條款」。
   Fixed-Size 會在條款中間切斷（例如第四條切成三個 chunk），
   查詢「違約金上限」時需要拼接多個 chunk，
   而且 Embedding 模型不知道這幾個 chunk 是同一條款的。
   Parent-Child 的設計：Child 用於精確搜尋定位，
   命中後換成 Parent（完整條款）給 LLM，
   LLM 看到的是完整語意單位，答案更準確。

Q: Metadata 為什麼要設計這麼細？
A: 大部分的合約查詢都有明確的過濾條件：
   「2026 年」「服務合約」「金額超過 X」。
   如果只有向量搜尋，需要從所有合約中搜，
   不僅慢，而且會找到不相關年度的合約。
   Metadata 過濾把向量搜尋的範圍縮小 10-100 倍，
   同時讓查詢更精確。
   設計原則：任何用戶可能 filter 的維度都要成為 metadata。

Q: 為什麼要做結構化提取 + 驗證？
A: 因為後續的分析（比較違約金、過濾金額）需要可計算的數值。
   如果只存原文，「NTD 1,500,000」和「壹佰伍拾萬」
   在程式碼層無法比較大小。
   驗證是因為 LLM 有時候漏欄位或金額算錯，
   不驗證就存進 DB，後面查詢會出錯。
═══════════════════════════════════════════════════════
""")